# Hadith Retriever — Run on pre-generated stories

This notebook takes a CSV of already-generated stories (with a built `retrieval_query` column)
and runs the **exact same retrieval logic from the original pipeline notebook** on each row:

**BGE-M3 top-50 → CrossEncoder (bge-reranker-base) rerank → Arabic-aware deduplication → top-10**

It does NOT call any LLM. No story generation, no summarization, no judge.
This makes retrieval experiments fast and reproducible.

Each input row is run twice and saved into the same output CSV:
1. **WITH summary** — using the existing `retrieval_query` column (topic + moral + summary).
2. **WITHOUT summary** — rebuilding the query from `input_topic` + `input_moral` only.

This way you can directly compare the effect of the summary on retrieval quality.


## Cell 1 — Setup + paths

In [ ]:
# ============================================================
# Cell 1 — Setup + paths
# Run once per session
# ============================================================

!pip install -q sentence-transformers pandas==2.2.2

from google.colab import drive
drive.mount("/content/drive")

import os
import re
import csv
import json
import gc
from datetime import datetime
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import torch

# ============================================================
# Paths
# ============================================================

BASE_DIR = "/content/drive/MyDrive/Islamic_Stories_Project/Islamic_Stories_Project"

# Inputs
STORIES_CSV      = f"{BASE_DIR}/stories_for_retriever.csv"
HADITH_META_CSV  = f"{BASE_DIR}/hadith_embedding/hadith_meta_bge.csv"
HADITH_EMBEDS_PT = f"{BASE_DIR}/hadith_embedding/hadith_embeds_bge.pt"

# Outputs — one CSV with both conditions side by side
RETRIEVER_OUTPUT_CSV = f"{BASE_DIR}/retriever_results.csv"

# Retrieval settings — identical to the original notebook
INITIAL_K              = 50
FINAL_K                = 10
SIMILARITY_THRESHOLD   = 0.78
CONTAINMENT_THRESHOLD  = 0.75

# Verify files
for label, p in [
    ("Stories CSV", STORIES_CSV),
    ("Hadith meta CSV", HADITH_META_CSV),
    ("Hadith embeddings", HADITH_EMBEDS_PT),
]:
    print(("✓" if os.path.exists(p) else "✗ MISSING"), label, "->", p)

print()
print("Output CSV ->", RETRIEVER_OUTPUT_CSV)


## Cell 2 — Load Hadith corpus + BGE-M3 + CrossEncoder

Same as the original notebook — both retrieval models run on CPU.

In [ ]:
# ============================================================
# Cell 2 — Load Hadith corpus + retrieval models
# ============================================================

from sentence_transformers import SentenceTransformer, CrossEncoder, util

# Hadith corpus
hadith_df = pd.read_csv(HADITH_META_CSV)
hadith_texts = hadith_df["text_ar"].astype(str).tolist()
hadith_embeds = torch.load(HADITH_EMBEDS_PT, map_location="cpu")
print(f"✓ Hadith corpus loaded: {len(hadith_df)} entries")

# BGE-M3 encoder (CPU)
embed_model = SentenceTransformer("BAAI/bge-m3").to("cpu")
print("✓ BGE-M3 loaded")

# Cross-encoder reranker (CPU)
reranker = CrossEncoder("BAAI/bge-reranker-base", device="cpu")
print("✓ bge-reranker-base loaded")

print("\nAll retrieval resources ready.")


## Cell 3 — Helpers (Arabic normalization, matn extraction, dedup)

These are copied verbatim from the original notebook — Arabic normalization for dedup only, matn extraction prefers quoted text, and dedup combines sequence similarity + word containment thresholds.

In [ ]:
# ============================================================
# Cell 3 — Helpers (identical to the original notebook)
# ============================================================

# ------------------------------------------------------------
# JSON-safe helper
# Fixes: TypeError: Object of type int64 is not JSON serializable
# ------------------------------------------------------------

def make_json_safe(obj):
    """Convert numpy / pandas values into normal Python types for JSON/CSV saving."""
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, tuple):
        return tuple(make_json_safe(v) for v in obj)
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj


def safe_get(row, col, default=""):
    """Read a value from a CSV row, returning default for missing/NaN."""
    if col not in row:
        return default
    val = row.get(col, default)
    try:
        if pd.isna(val):
            return default
    except Exception:
        pass
    return val


# ------------------------------------------------------------
# Arabic normalization — for duplicate detection ONLY
# Does NOT change Hadith text shown to the evaluator.
# ------------------------------------------------------------

def normalize_arabic(text):
    """Normalize Arabic text to make duplicate detection easier."""
    if text is None:
        return ""

    text = str(text)

    # Remove Arabic diacritics
    text = re.sub(r"[\u0617-\u061A\u064B-\u0652]", "", text)

    # Remove tatweel
    text = text.replace("ـ", "")

    # Normalize common Arabic letters
    text = re.sub("[إأآا]", "ا", text)
    text = text.replace("ى", "ي")
    text = text.replace("ة", "ه")
    text = text.replace("ؤ", "و")
    text = text.replace("ئ", "ي")

    # Remove punctuation and symbols
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)

    # Collapse spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


def clean_spaces(text):
    """Clean extra spaces before matn extraction."""
    if text is None:
        return ""
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text


def extract_hadith_matn(text):
    """
    Try to extract the quoted Hadith matn from the full Hadith text.
    If the Hadith contains quoted text, use the longest quoted segment.
    Otherwise, use the full text.
    """
    if text is None:
        return ""

    text = str(text)

    patterns = [
        r"‏\"‏(.*?)‏\"‏",
        r'"(.*?)"',
        r"“(.*?)”",
        r"«(.*?)»",
    ]

    matches = []
    for pat in patterns:
        found = re.findall(pat, text, flags=re.DOTALL)
        matches.extend(found)

    if matches:
        matn = max(matches, key=len)
        return clean_spaces(matn)

    return clean_spaces(text)


def candidate_dedup_key(candidate):
    """
    Build comparison text for duplicate detection.
    Prefer matn if extractable (and long enough); otherwise use full Hadith text.
    Arabic normalization is applied here only.
    """
    text = candidate.get("text_ar", "")
    matn = extract_hadith_matn(text)

    # If extracted matn is too short, fallback to full text
    if len(matn.split()) < 4:
        matn = clean_spaces(text)

    return normalize_arabic(matn)


# ------------------------------------------------------------
# Stronger deduplication (same thresholds as original notebook)
# ------------------------------------------------------------

def deduplicate_after_crossencoder(
    candidates,
    similarity_threshold=0.78,
    containment_threshold=0.75,
    final_k=10,
):
    """
    Remove near-duplicate Hadith candidates AFTER CrossEncoder reranking.

    A candidate is removed if either:
    1. Sequence similarity with an already-kept Hadith is high.
    2. Most of its words overlap with an already-kept Hadith.

    Why after CE?
    CrossEncoder first picks the best wording/version.
    Then dedup removes repeated narrations while keeping the highest-ranked version.
    """
    unique = []
    unique_keys = []

    for cand in candidates:
        cand_key = candidate_dedup_key(cand)

        if not cand_key:
            continue

        cand_words = set(cand_key.split())
        is_duplicate = False

        for kept_key in unique_keys:
            kept_words = set(kept_key.split())

            # 1. Sequence-level similarity
            seq_sim = SequenceMatcher(None, cand_key, kept_key).ratio()

            # 2. Word containment similarity
            if len(cand_words) == 0 or len(kept_words) == 0:
                containment = 0
            else:
                overlap = len(cand_words.intersection(kept_words))
                containment = overlap / min(len(cand_words), len(kept_words))

            if seq_sim >= similarity_threshold or containment >= containment_threshold:
                is_duplicate = True
                break

        if not is_duplicate:
            unique.append(cand)
            unique_keys.append(cand_key)

        if len(unique) >= final_k:
            break

    # Reassign ranks after deduplication
    for rank, cand in enumerate(unique, start=1):
        cand["rank_after_dedup"] = rank

    return unique


print("✓ Helpers ready")


## Cell 4 — Retriever function (BGE → CrossEncoder → Dedup)

Identical to `retrieve_bge_ce_dedup` from the original notebook.

In [ ]:
# ============================================================
# Cell 4 — Retriever: BGE top-50 → CrossEncoder → Dedup → top-10
# ============================================================

def retrieve_bge_ce_dedup(
    retrieval_query,
    initial_k=INITIAL_K,
    final_k=FINAL_K,
    similarity_threshold=SIMILARITY_THRESHOLD,
    containment_threshold=CONTAINMENT_THRESHOLD,
):
    """
    Final retriever logic:
    1. BGE retrieves top-N (default 50).
    2. CrossEncoder reranks all candidates.
    3. Dedup removes repeated / near-duplicate narrations.
    4. Return final top-K (default 10).
    """

    # -------- BGE retrieval --------
    q_emb = embed_model.encode(
        retrieval_query,
        convert_to_tensor=True,
        normalize_embeddings=True,
        device="cpu",
    )

    sims = util.cos_sim(q_emb, hadith_embeds)[0].cpu().tolist()

    top_idx = sorted(
        range(len(sims)),
        key=lambda i: sims[i],
        reverse=True,
    )[:initial_k]

    candidates = []
    for bge_rank, i in enumerate(top_idx, start=1):
        row_h = hadith_df.iloc[i]
        candidates.append({
            "idx":       int(i),
            "rank_bge":  bge_rank,
            "score_bge": float(sims[i]),
            "text_ar":   str(hadith_texts[i]),
            "source":    make_json_safe(row_h.get("source")),
            "chapter":   make_json_safe(row_h.get("chapter")),
            "hadith_id": make_json_safe(row_h.get("hadith_id")),
        })

    # -------- CrossEncoder reranking --------
    if candidates:
        pairs = [(retrieval_query, c["text_ar"]) for c in candidates]
        ce_scores = reranker.predict(pairs).tolist()

        for c, s in zip(candidates, ce_scores):
            c["score_ce"] = float(s)

        reranked = sorted(candidates, key=lambda c: c["score_ce"], reverse=True)

        for rank, c in enumerate(reranked, start=1):
            c["rank_after_ce"] = rank
    else:
        reranked = []

    # -------- Dedup AFTER CrossEncoder --------
    final_candidates = deduplicate_after_crossencoder(
        reranked,
        similarity_threshold=similarity_threshold,
        containment_threshold=containment_threshold,
        final_k=final_k,
    )

    selected = final_candidates[0] if final_candidates else None

    return {
        "initial_k": initial_k,
        "final_k": final_k,
        "similarity_threshold": similarity_threshold,
        "containment_threshold": containment_threshold,
        "bge_candidates":      candidates,
        "reranked_candidates": reranked,
        "final_candidates":    final_candidates,
        "selected":            selected,
    }


def build_query_without_summary(topic, moral):
    """Rebuild a retrieval query from topic + moral only (no summary)."""
    return (
        f"القيمة الإسلامية: {topic}\n"
        f"العبرة من القصة: {moral}"
    )


print("✓ Retriever ready")


## Cell 5 — CSV writer for retriever results

One row per input story. Contains both the **with-summary** and **without-summary** retrieval results side by side so the comparison is easy.

In [ ]:
# ============================================================
# Cell 5 — CSV writer for retriever results
# ============================================================

RETRIEVER_COLUMNS = [
    "sample_id",
    "timestamp",

    # Story inputs (carried over from the stories CSV)
    "input_age",
    "input_topic",
    "input_moral",
    "input_place",
    "input_country",
    "input_season",
    "input_activity",
    "input_emotion",
    "input_dialogue",
    "input_plot_twist",
    "input_end_of_story",

    "generated_story",
    "extracted_moral",

    # Retrieval settings (recorded for reproducibility)
    "initial_k",
    "final_k",
    "dedup_similarity_threshold",
    "dedup_containment_threshold",

    # WITH-summary retrieval
    "with_summary_query",
    "with_summary_top_k_candidates",
    "with_summary_selected_hadith",
    "with_summary_hadith_source",
    "with_summary_hadith_chapter",
    "with_summary_hadith_id",
    "with_summary_bge_score",
    "with_summary_cross_encoder_score",

    # WITHOUT-summary retrieval
    "no_summary_query",
    "no_summary_top_k_candidates",
    "no_summary_selected_hadith",
    "no_summary_hadith_source",
    "no_summary_hadith_chapter",
    "no_summary_hadith_id",
    "no_summary_bge_score",
    "no_summary_cross_encoder_score",
]


def _serialize_candidates(final_candidates):
    safe = make_json_safe([
        {
            "rank_after_dedup": c.get("rank_after_dedup"),
            "rank_after_ce":    c.get("rank_after_ce"),
            "rank_bge":         c.get("rank_bge"),
            "idx":              c.get("idx"),
            "text_ar":          c.get("text_ar"),
            "source":           c.get("source"),
            "chapter":          c.get("chapter"),
            "hadith_id":        c.get("hadith_id"),
            "score_bge":        c.get("score_bge"),
            "score_ce":         c.get("score_ce"),
        }
        for c in final_candidates
    ])
    return json.dumps(safe, ensure_ascii=False)


def _csv_has_header(path):
    if not os.path.exists(path):
        return False
    try:
        with open(path, "r", encoding="utf-8-sig", newline="") as f:
            first = f.readline()
        return first.strip().startswith("sample_id")
    except Exception:
        return False


def append_retriever_row(path, source_row, sample_id, with_result, no_result):
    """Save one row containing both retrieval conditions."""

    parent = os.path.dirname(os.path.abspath(path))
    if parent:
        os.makedirs(parent, exist_ok=True)

    write_header = not _csv_has_header(path)
    mode = "a" if os.path.exists(path) else "w"

    with_selected = with_result.get("selected") or {}
    no_selected   = no_result.get("selected")   or {}

    row = {
        "sample_id": sample_id,
        "timestamp": datetime.now().isoformat(timespec="seconds"),

        "input_age":          safe_get(source_row, "input_age"),
        "input_topic":        safe_get(source_row, "input_topic"),
        "input_moral":        safe_get(source_row, "input_moral"),
        "input_place":        safe_get(source_row, "input_place"),
        "input_country":      safe_get(source_row, "input_country"),
        "input_season":       safe_get(source_row, "input_season"),
        "input_activity":     safe_get(source_row, "input_activity"),
        "input_emotion":      safe_get(source_row, "input_emotion"),
        "input_dialogue":     safe_get(source_row, "input_dialogue"),
        "input_plot_twist":   safe_get(source_row, "input_plot_twist"),
        "input_end_of_story": safe_get(source_row, "input_end_of_story"),

        "generated_story": safe_get(source_row, "generated_story"),
        "extracted_moral": safe_get(source_row, "extracted_moral"),

        "initial_k":                   with_result.get("initial_k"),
        "final_k":                     with_result.get("final_k"),
        "dedup_similarity_threshold":  with_result.get("similarity_threshold"),
        "dedup_containment_threshold": with_result.get("containment_threshold"),

        # WITH summary
        "with_summary_query":               safe_get(source_row, "retrieval_query"),
        "with_summary_top_k_candidates":    _serialize_candidates(with_result.get("final_candidates") or []),
        "with_summary_selected_hadith":     with_selected.get("text_ar", ""),
        "with_summary_hadith_source":       with_selected.get("source", ""),
        "with_summary_hadith_chapter":      with_selected.get("chapter", ""),
        "with_summary_hadith_id":           with_selected.get("hadith_id", ""),
        "with_summary_bge_score":           with_selected.get("score_bge", ""),
        "with_summary_cross_encoder_score": with_selected.get("score_ce", ""),

        # WITHOUT summary
        "no_summary_query":                 build_query_without_summary(
                                                safe_get(source_row, "input_topic"),
                                                safe_get(source_row, "input_moral"),
                                            ),
        "no_summary_top_k_candidates":      _serialize_candidates(no_result.get("final_candidates") or []),
        "no_summary_selected_hadith":       no_selected.get("text_ar", ""),
        "no_summary_hadith_source":         no_selected.get("source", ""),
        "no_summary_hadith_chapter":        no_selected.get("chapter", ""),
        "no_summary_hadith_id":             no_selected.get("hadith_id", ""),
        "no_summary_bge_score":             no_selected.get("score_bge", ""),
        "no_summary_cross_encoder_score":   no_selected.get("score_ce", ""),
    }

    with open(path, mode, encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=RETRIEVER_COLUMNS)
        if write_header:
            writer.writeheader()
        writer.writerow(row)

    return row


print("✓ CSV writer ready")


## Cell 6 — Run retrieval on all rows in the stories CSV

For each story, run retrieval twice (with and without summary) and save both into one CSV row.

Set `OVERWRITE_OUTPUT=True` to clear the previous output and start a fresh file.

In [ ]:
# ============================================================
# Cell 6 — Batch retrieval over all stories in the CSV
# ============================================================

OVERWRITE_OUTPUT = True   # set False to append instead of starting fresh

if not os.path.exists(STORIES_CSV):
    raise FileNotFoundError(f"Stories CSV not found: {STORIES_CSV}")

stories_df = pd.read_csv(STORIES_CSV)
print("=" * 80)
print(f"Stories file: {STORIES_CSV}")
print(f"Total stories: {len(stories_df)}")
print(f"Output file:   {RETRIEVER_OUTPUT_CSV}")
print("=" * 80)

if OVERWRITE_OUTPUT and os.path.exists(RETRIEVER_OUTPUT_CSV):
    os.remove(RETRIEVER_OUTPUT_CSV)
    print("Old retriever output deleted. Starting fresh.\n")

for idx, source_row in stories_df.iterrows():
    sample_id = idx + 1

    topic = safe_get(source_row, "input_topic")
    moral = safe_get(source_row, "input_moral")

    print("\n" + "=" * 80)
    print(f"Sample {sample_id}/{len(stories_df)}")
    print(f"  topic: {topic}")
    print(f"  moral: {moral}")
    print("=" * 80)

    # ---- WITH summary: use the query already in the CSV ----
    with_query = safe_get(source_row, "retrieval_query")
    print("\n[WITH summary] query:")
    print(with_query)

    with_result = retrieve_bge_ce_dedup(with_query)
    with_sel = with_result.get("selected") or {}
    print(f"[WITH summary]  selected hadith_id={with_sel.get('hadith_id')}  "
          f"BGE={with_sel.get('score_bge', 0):.4f}  CE={with_sel.get('score_ce', 0):.4f}")

    # ---- WITHOUT summary: rebuild query from topic + moral only ----
    no_query = build_query_without_summary(topic, moral)
    print("\n[NO summary] query:")
    print(no_query)

    no_result = retrieve_bge_ce_dedup(no_query)
    no_sel = no_result.get("selected") or {}
    print(f"[NO summary]    selected hadith_id={no_sel.get('hadith_id')}  "
          f"BGE={no_sel.get('score_bge', 0):.4f}  CE={no_sel.get('score_ce', 0):.4f}")

    # ---- Save both into one row ----
    append_retriever_row(
        path=RETRIEVER_OUTPUT_CSV,
        source_row=source_row,
        sample_id=sample_id,
        with_result=with_result,
        no_result=no_result,
    )
    print("\n  ✓ saved")

print("\n" + "=" * 80)
print("Batch retrieval completed.")
print("Output CSV:", RETRIEVER_OUTPUT_CSV)
print("=" * 80)


## Cell 7 — Quick comparison view

Shows a side-by-side table of the selected Hadith for each sample under both conditions, so you can scan for differences.

In [ ]:
import pandas as pd
from IPython.display import display

df = pd.read_csv(RETRIEVER_OUTPUT_CSV)
print(f"Total rows: {len(df)}\n")

# Compact comparison table
compare = df[[
    "sample_id",
    "input_topic",
    "input_moral",
    "with_summary_hadith_id",
    "with_summary_cross_encoder_score",
    "no_summary_hadith_id",
    "no_summary_cross_encoder_score",
]].copy()

# Did the same hadith get picked under both conditions?
compare["same_top_hadith"] = (
    compare["with_summary_hadith_id"].astype(str)
    == compare["no_summary_hadith_id"].astype(str)
)

display(compare)

print()
print(f"Same top hadith picked in {compare['same_top_hadith'].sum()}/{len(compare)} samples.")


## Cell 8 — Inspect one sample in detail

Change `SAMPLE_ID` to look at the full top-10 list (both conditions) for a specific story.

In [ ]:
SAMPLE_ID = 1   # change to inspect a different sample

df = pd.read_csv(RETRIEVER_OUTPUT_CSV)
row = df[df["sample_id"] == SAMPLE_ID].iloc[0]

print("=" * 80)
print(f"Sample {SAMPLE_ID}")
print("=" * 80)
print("Topic:", row["input_topic"])
print("Moral:", row["input_moral"])
print()
print("Extracted moral from story:")
print(row["extracted_moral"])

for label, q_col, top_col in [
    ("WITH summary",   "with_summary_query",   "with_summary_top_k_candidates"),
    ("NO  summary",    "no_summary_query",     "no_summary_top_k_candidates"),
]:
    print("\n" + "=" * 80)
    print(label)
    print("=" * 80)
    print("Query:")
    print(row[q_col])
    print()
    cands = json.loads(row[top_col])
    print(f"Top-{len(cands)} after CrossEncoder + dedup:")
    for c in cands:
        print("\n" + "-" * 60)
        print(f"  rank_after_dedup : {c.get('rank_after_dedup')}")
        print(f"  rank_after_ce    : {c.get('rank_after_ce')}")
        print(f"  rank_bge         : {c.get('rank_bge')}")
        print(f"  hadith_id        : {c.get('hadith_id')}")
        print(f"  source           : {c.get('source')}")
        print(f"  chapter          : {c.get('chapter')}")
        print(f"  BGE score        : {c.get('score_bge', 0):.4f}")
        print(f"  CE  score        : {c.get('score_ce', 0):.4f}")
        print("  text:")
        print(" " * 4 + str(c.get("text_ar", ""))[:500])
